# 18 · Bit Manipulation

> Operazioni sui singoli bit di un intero: leggere, accendere, spegnere, contare. Il trucco più usato è lo **XOR** (`x ^ x == 0`). In Python gli interi non hanno dimensione fissa, quindi i problemi "a 32 bit" richiedono una **maschera**.
> Idee per i singoli problemi: [Pattern algoritmici › Bit manipulation](../Fundamentals/Pattern%20algoritmici.md#bit-manipulation-essenziale)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Single Number | le coppie si annullano con XOR | `result ^= x` |
| Number of 1 Bits | conta i bit accesi | `n & (n - 1)` spegne l'ultimo bit, `n.bit_count()` |
| Counting Bits | `bits(i) = bits(i >> 1) + (i & 1)` | DP su `range(n + 1)` |
| Reverse Bits | estrai il bit più basso, inseriscilo in fondo al risultato, 32 volte | `(n >> i) & 1`, `result << 1 \| bit` |
| Missing Number | XOR di indici e valori, oppure somma attesa − somma | `n * (n + 1) // 2` |
| Sum of Two Integers | somma senza `+`: XOR = somma senza riporto, `&` + shift = riporto | maschera `0xFFFFFFFF`, conversione a negativo |
| Reverse Integer | cifre al contrario con controllo dei limiti a 32 bit | `2**31 - 1`, `-(2**31)`, segno a parte |

## Sintassi al volo

| Operatore | Significato | Esempio |
|---|---|---|
| `a & b` | AND: 1 se entrambi | `0b1100 & 0b1010 == 0b1000` |
| `a \| b` | OR: 1 se almeno uno | `0b1100 \| 0b1010 == 0b1110` |
| `a ^ b` | XOR: 1 se diversi | `0b1100 ^ 0b1010 == 0b0110` |
| `~a` | NOT: in Python `~a == -a - 1` | `~5 == -6` |
| `a << k` | shift a sinistra: `a * 2**k` | `3 << 2 == 12` |
| `a >> k` | shift a destra: `a // 2**k` | `13 >> 1 == 6` |

| Serve | Sintassi |
|---|---|
| bit `i` acceso? | `(n >> i) & 1` |
| accendi / spegni / inverti il bit `i` | `n \| (1 << i)` · `n & ~(1 << i)` · `n ^ (1 << i)` |
| spegni il bit acceso più basso | `n & (n - 1)` |
| isola il bit acceso più basso | `n & -n` |
| potenza di 2? | `n > 0 and n & (n - 1) == 0` |
| conta i bit accesi | `n.bit_count()` (3.10+) · `bin(n).count("1")` |
| rappresentazione | `bin(10) == "0b1010"` · `f"{10:08b}" == "00001010"` · `int("1010", 2) == 10` |

## Leggere i bit

In [1]:
n = 0b1011  # 11
print(bin(n), f"{n:08b}", int("1011", 2))  # -> 0b1011 00001011 11
print([(n >> i) & 1 for i in range(4)])     # -> [1, 1, 0, 1]: dal meno significativo
print(n & 1, n % 2)                          # -> 1 1: dispari

0b1011 00001011 11
[1, 1, 0, 1]
1 1


## XOR: le coppie si annullano

Proprietà: `x ^ x == 0`, `x ^ 0 == x`, ed è commutativo e associativo. Se ogni valore compare due volte tranne uno, lo XOR di tutto lascia solo quello.

In [2]:
from functools import reduce
from operator import xor

order_ids = [4, 1, 2, 1, 2]
single = 0
for x in order_ids:
    single ^= x
print(single, reduce(xor, order_ids))  # -> 4 4

# Valore mancante in 0..n: XOR di tutti gli indici e di tutti i valori
values = [3, 0, 1]
missing = len(values)
for i, v in enumerate(values):
    missing ^= i ^ v
print(missing, len(values) * (len(values) + 1) // 2 - sum(values))  # -> 2 2

4 4
2 2


## `n & (n - 1)`: spegnere il bit più basso

Sottrarre 1 inverte il bit acceso più basso e tutti gli zeri alla sua destra; l'AND li azzera. Ripetere finché `n` è 0 conta i bit accesi in tanti passi quanti sono i bit (non 32).

In [3]:
n = 0b10110
print(f"{n:05b} {n - 1:05b} {n & (n - 1):05b}")  # -> 10110 10101 10100

count = 0
while n:
    n &= n - 1
    count += 1
print(count, (0b10110).bit_count())  # -> 3 3

10110 10101 10100
3 3


## DP sui bit

`i >> 1` è `i` senza l'ultimo bit, già calcolato perché più piccolo: `bits[i] = bits[i >> 1] + (i & 1)`.

In [4]:
bits = [0] * 9
for i in range(1, 9):
    bits[i] = bits[i >> 1] + (i & 1)
print(bits)  # -> [0, 1, 1, 2, 1, 2, 2, 3, 1]

[0, 1, 1, 2, 1, 2, 2, 3, 1]


## Costruire un numero bit per bit

Per invertire l'ordine dei bit: a ogni passo sposta il risultato a sinistra e aggiungi il bit più basso dell'input, poi sposta l'input a destra. Esattamente 32 iterazioni, anche se `n` diventa 0 prima (gli zeri iniziali contano).

In [5]:
n = 0b1011  # su 4 bit per leggibilità
result = 0
for _ in range(4):
    result = (result << 1) | (n & 1)
    n >>= 1
print(f"{result:04b}")  # -> 1101

1101


## Interi a 32 bit in Python

Gli interi Python hanno precisione arbitraria: `-1` non è `0xFFFFFFFF` e i cicli che si basano sull'overflow non terminano. Si usa una maschera per restare su 32 bit e alla fine si riconverte a negativo se il bit di segno (bit 31) è acceso.

In [6]:
MASK = 0xFFFFFFFF       # 32 bit a 1
MAX_INT = 0x7FFFFFFF    # 2**31 - 1

x = -3 & MASK
print(hex(x), x)        # -> 0xfffffffd 4294967293: la vista "unsigned" di -3

def to_signed(v: int) -> int:
    return v if v <= MAX_INT else ~(v ^ MASK)  # equivale a v - 2**32

print(to_signed(x))     # -> -3

a, b = 5, 3
print(a ^ b, (a & b) << 1)  # -> 6 2: somma senza riporto e riporto (6 + 2 = 8)

0xfffffffd 4294967293
-3
6 2


## Limiti a 32 bit con numeri decimali

Per problemi come *Reverse Integer*: gestisci il segno a parte, lavora sul valore assoluto, e controlla che il risultato stia in `[-2**31, 2**31 - 1]`.

In [7]:
INT_MIN, INT_MAX = -(2**31), 2**31 - 1
print(INT_MIN, INT_MAX)  # -> -2147483648 2147483647

x = -1230
sign = -1 if x < 0 else 1
reversed_abs = int(str(abs(x))[::-1])  # "0321" -> 321
result = sign * reversed_abs
print(result, INT_MIN <= result <= INT_MAX)  # -> -321 True

print(-7 % 10, -7 // 10)  # -> 3 -1: con i negativi % e // non fanno quello che faresti in C

-2147483648 2147483647
-321 True
3 -1


## Errori comuni

- Precedenza: `+`/`-` legano più degli shift, quindi `1 << k - 1` è `1 << (k - 1)` e `a + b << 1` è `(a + b) << 1`. Nel dubbio, parentesi. (A differenza di C, in Python `n & 1 == 0` funziona: `&` lega più di `==`.)
- Cicli `while b != 0` con numeri negativi senza maschera: non terminano.
- `~x` in Python è `-x - 1`, non il complemento a 32 bit.
- Reverse Bits: fermarsi quando `n == 0` invece di fare sempre 32 iterazioni.
- `%` e `//` con negativi arrotondano verso -∞: estrai le cifre dal valore assoluto.